# Download Videos_L21_a.zip\n\nNotebook n?y ch? t?i `https://aic-data.ledo.io.vn/Videos_L21_a.zip` v? chu?n b? output ?? t?o Kaggle Dataset.

In [ ]:
%pip install -q requests tqdm

In [ ]:
from __future__ import annotations

import hashlib
import json
import shutil
import time
import zipfile
from pathlib import Path

import requests
from tqdm.auto import tqdm

FILE_NAME = 'Videos_L21_a.zip'
URL = 'https://aic-data.ledo.io.vn/Videos_L21_a.zip'
OUTPUT_ROOT = Path('/kaggle/working/dataset')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FINAL_PATH = OUTPUT_ROOT / FILE_NAME
PART_PATH = OUTPUT_ROOT / f'{FILE_NAME}.part'
CHUNK_SIZE = 8 * 1024 * 1024
MAX_RETRIES = 8


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(CHUNK_SIZE), b''):
            digest.update(chunk)
    return digest.hexdigest()


def download() -> dict:
    if FINAL_PATH.exists() and FINAL_PATH.stat().st_size > 1_000_000:
        print('File ?? t?n t?i, ki?m tra l?i ZIP:', FINAL_PATH)
    else:
        last_error = None
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                offset = PART_PATH.stat().st_size if PART_PATH.exists() else 0
                headers = {'User-Agent': 'Mozilla/5.0 AIC-Kaggle-Downloader/1.0'}
                if offset:
                    headers['Range'] = f'bytes={offset}-'
                response = requests.get(URL, headers=headers, stream=True, timeout=(30, 180))
                if offset and response.status_code == 200:
                    response.close()
                    PART_PATH.unlink(missing_ok=True)
                    offset = 0
                    response = requests.get(URL, headers={'User-Agent': headers['User-Agent']}, stream=True, timeout=(30, 180))
                response.raise_for_status()
                mode = 'ab' if offset and response.status_code == 206 else 'wb'
                if mode == 'wb':
                    offset = 0
                content_range = response.headers.get('Content-Range', '')
                total_text = content_range.split('/')[-1]
                total = int(total_text) if total_text.isdigit() else None
                if total is None and response.headers.get('Content-Length'):
                    total = offset + int(response.headers['Content-Length'])
                with PART_PATH.open(mode) as handle, tqdm(
                    total=total, initial=offset, unit='B', unit_scale=True, desc=FILE_NAME
                ) as progress:
                    for chunk in response.iter_content(CHUNK_SIZE):
                        if chunk:
                            handle.write(chunk)
                            progress.update(len(chunk))
                response.close()
                PART_PATH.replace(FINAL_PATH)
                break
            except Exception as exc:
                last_error = str(exc)
                print(f'Attempt {attempt}/{MAX_RETRIES} failed: {last_error}')
                time.sleep(min(60, attempt * 5))
        else:
            raise RuntimeError(f'Kh?ng t?i ???c {FILE_NAME}: {last_error}')

    with zipfile.ZipFile(FINAL_PATH) as archive:
        bad_member = archive.testzip()
        members = len(archive.infolist())
    if bad_member:
        raise RuntimeError(f'ZIP h?ng t?i file: {bad_member}')
    result = {
        'name': FILE_NAME,
        'url': URL,
        'bytes': FINAL_PATH.stat().st_size,
        'sha256': sha256_file(FINAL_PATH),
        'members': members,
        'status': 'downloaded',
    }
    (OUTPUT_ROOT / 'manifest.json').write_text(
        json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    return result


total, used, free = shutil.disk_usage('/kaggle/working')
print(f'Free disk: {free / 1024**3:.2f} GiB')
result = download()
print(json.dumps(result, ensure_ascii=False, indent=2))
print('Save Version -> Save output -> Create Dataset')
